## Reproductible fitting for the `jmstate` package

In [ ]:
%pip install jmstate matplotlib pandas

In [ ]:
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch

In [ ]:
import re


def plot_history(
    vals: list[torch.Tensor], true_val: torch.Tensor, name: str, colors: list[Any]
):
    plt.plot(torch.cat(vals, dim=0))

    for i, t in enumerate(true_val):
        plt.axhline(
            y=t.item(),
            color=colors[i % len(colors)],
            linestyle="--",
            label=(
                f"True value of {name}" + (f"({i + 1})" if true_val.numel() > 1 else "")
            ),
        )

    plt.title(f"Stochastic optimization of the parameter {name}")
    plt.legend(fontsize=11)
    plt.xlabel("Iteration")
    plt.ylabel("Value")
    plt.tight_layout()
    plt.savefig(f"../figures/{re.sub(r'[^A-Za-z0-9]+', '', name)}.pdf")
    plt.show()


def param_names(params_dict: dict[Any, Any]) -> list[str]:
    names: list[str] = []

    def _rec(node: dict[Any, Any] | torch.Tensor, prefix: list[Any]):
        if isinstance(node, torch.Tensor):
            base_name = str(prefix[0]) + "".join(f"[{item}]" for item in prefix[1:])
            num_elements = node.numel()
            if num_elements <= 1:
                names.append(base_name)
            else:
                names.extend(f"{base_name}[{i + 1}]" for i in range(num_elements))
        else:
            for k, v in node.items():
                prefix.append(k)
                _rec(v, prefix)
                prefix.pop()

    _rec(params_dict, [])

    return names

In [ ]:
torch.manual_seed(42)


TAU = 6.0


def reg(t: torch.Tensor, psi: torch.Tensor):
    b, w1, w2 = psi.chunk(3, dim=-1)  # Extract relevant terms

    # psi has shape (n_chains, n_individuals, n_repetitions)
    return (b + w1 * t + (w2 - w1) * (t > TAU) * (t - TAU)).unsqueeze(-1)


def link(t: torch.Tensor, psi: torch.Tensor):
    b, w1, w2 = psi.chunk(3, dim=-1)

    diff = (w2 - w1) * (t > TAU)
    val = b + w1 * t + diff * (t - TAU)
    der = w1 + diff
    return torch.cat([val.unsqueeze(-1), der.unsqueeze(-1)], dim=-1)


def random_far_apart(
    n: int, m: int, a: torch.Tensor, b: torch.Tensor, min_dist: torch.Tensor
):
    L_free = (b - a) - (m - 1) * min_dist

    y = torch.rand(n, m) * L_free
    y, _ = torch.sort(y, dim=1)

    gap_offset = torch.arange(m) * min_dist

    return a + y + gap_offset

In [ ]:
from jmstate.functions import Exponential, gamma_plus_b
from jmstate.typedefs import ModelDesign

# Survival model specification
surv = {
    (0, 1): (Exponential(0.1), link),
    (0, 2): (Exponential(0.01), link),
    (1, 2): (Exponential(0.2), link),
}

# Model design gathers regression, link and hazard functions
model_design = ModelDesign(gamma_plus_b, reg, surv)

In [ ]:
from jmstate.typedefs import ModelParams
from jmstate.utils import repr_from_cov

# Gaussian means
gamma = torch.tensor([2.5, -1.3, 0.2])

# Covariance matrices
Q = torch.diag(torch.tensor([0.6, 0.2, 0.3]))
R = torch.tensor([[1.7]])

# Link parameters
alphas = {
    (0, 1): torch.tensor([-0.5, -3.0]),
    (0, 2): torch.tensor([-1.0, -5.0]),
    (1, 2): torch.tensor([0.0, -1.2]),
}

# Covariate parameters
betas = {
    (0, 1): torch.tensor([-1.3]),
    (0, 2): torch.tensor([-0.9]),
    (1, 2): torch.tensor([-0.7]),
}

# Instance declaration
real_params = ModelParams(
    gamma,
    repr_from_cov(Q, method="diag"),
    repr_from_cov(R, method="ball"),
    alphas,
    betas,
)

In [ ]:
from jmstate import MultiStateJointModel
from jmstate.typedefs import ModelData, SampleData
from torch.distributions import MultivariateNormal

# Declare the true underlying model
real_model = MultiStateJointModel(model_design, real_params)


def gen_data(n: int, m: int):
    # Censoring times
    c = torch.rand(n, 1) * 5 + 10

    # Covariates
    x = torch.randn(n, 1)

    # Latent and noise distributions
    Q_dist = MultivariateNormal(torch.zeros(Q.size(0)), Q)
    R_dist = MultivariateNormal(torch.zeros(R.size(0)), R)

    # Individual effects
    b = Q_dist.sample((n,))
    psi = model_design.individual_effects_fn(gamma, x, b)

    # Generates random evaluations points with a minimum distance
    a = torch.zeros((n, 1))
    b = torch.full((n, 1), 15)
    t = random_far_apart(n, m, a, b, 0.7 * b / m)

    # Defines initial state for individuals
    trajectories_init = [[(0.0, 0)]] * n

    # Samples trajectories
    sample_data = SampleData(x, trajectories_init, psi)
    trajectories = real_model.sample_trajectories(sample_data, c)

    # Samples longitudinal values
    y = model_design.regression_fn(t, psi)
    y += R_dist.sample(y.shape[:2])

    # Censors longitudinal measurements exceeding censoring times
    y[t > c] = torch.nan

    return x, t, y, trajectories, c


# Generate data
data = ModelData(*gen_data(1000, 20))

In [ ]:
plt.plot(data.t[:50].T, data.y[:50].squeeze(-1).T)
plt.title("Longitudinal sample")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("../figures/simulated-sample.pdf")
plt.show()

In [ ]:
from jmstate.utils import build_buckets

buckets = build_buckets(data.trajectories)
print({key: val.idxs.numel() for key, val in buckets.items()})

In [ ]:
# Declares initial parameters; zero mean and unit variance
init_params = ModelParams(
    torch.zeros_like(gamma),
    repr_from_cov(torch.eye(Q.size(0)), method="diag"),
    repr_from_cov(torch.eye(R.size(0)), method="ball"),
    {k: torch.zeros_like(v) for k, v in alphas.items()},
    {k: torch.zeros_like(v) for k, v in betas.items()},
)

In [ ]:
from jmstate.jobs import Fit, LogParamsHistory, ParamStop

# Declares initial model
model = MultiStateJointModel(model_design, init_params)

# Runs optimization process
metrics = model.do(
    data,
    job_factories=[
        Fit(lr=0.5, fused=True),
        ParamStop(rtol=0.1),
        LogParamsHistory(),  # Returns a list of ModelParams
    ],
    max_iterations=500,
)

In [ ]:
params_dict = model.params_.as_dict
real_params_dict = real_model.params_.as_dict
colors = plt.get_cmap("tab10").colors

for key, val in params_dict.items():
    if isinstance(val, torch.Tensor):
        history = [p.as_dict[key][None] for p in next(iter(vars(metrics).values()))]
        plot_history(history, real_params_dict[key], key, colors)
    else:
        for subkey in val:
            history = [
                p.as_dict[key][subkey][None] for p in next(iter(vars(metrics).values()))
            ]
            plot_history(
                history,
                real_params_dict[key][subkey],
                key + f"[{subkey}]",
                colors,
            )

In [ ]:
params_list: list[torch.Tensor] = []
n_runs = 100

for _ in range(n_runs):
    data = ModelData(*gen_data(1000, 20))

    model = MultiStateJointModel(model_design, init_params)
    model.do(
        data,
        job_factories=[Fit(lr=0.5, fused=True), ParamStop(rtol=0.1)],
        max_iterations=500,
    )

    params_list.append(model.params_.as_flat_tensor.view(1, -1))

In [ ]:
names = param_names(real_model.params_.as_dict)

stacked_params = torch.cat(params_list, dim=0)
mean = stacked_params.mean(dim=0)
std = stacked_params.std(dim=0)
rmse = (std**2 + (mean - real_model.params_.as_flat_tensor) ** 2).sqrt()

data = {
    "True value": real_model.params_.as_flat_tensor.numpy(),
    "Mean": mean.numpy(),
    "Std error": std.numpy(),
    "RMSE": rmse.numpy(),
}

index = [f"{names[i]}" for i in range(mean.size(0))]

df = pd.DataFrame(data, index=index)
pd.set_option("display.float_format", "{:.4e}".format)

print(df)